# Description

W tym notatniku przeprowadzane są wszelkie eksperymenty, zarówno dla autoenkodera wariacyjnego i nie wariacyjnego, dla wszystkich członów funkcji straty, w wersji z douczaniem i bez (łącznie 12 eksperymentów)

# Imports

In [12]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar
import yaml
import sys
import os
import tqdm
import wandb
import json

sys.path.append('../')  # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.GraphAutoencoder import GraphAutoencoder
from src.models.KlejdaGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from pyprojroot import here

current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
from FramsticksLib import FramsticksLib
from deap import tools, algorithms
import yaml
from src.deap.deap_setup import prepare_native_toolbox, prepare_cmaes_toolbox
from src.deap.constraints import is_feasible_fitness_criteria
from src.deap.save_and_load_results import save_genotypes_json
from src.deap.custom_ea_algorithms import run_cma_es_with_validation
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from src.deap.AutoencoderEvaluator import AutoencoderEvaluator
import numpy as np
import frams


import time

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Pre-processing

In [13]:
project_dir = here()
# Przygotowanie checkpointów nauczonych autoenkoderów
checkpoints_dir = project_dir / 'notebooks' / 'checkpoints' / 'final_checkpoints' / 'klejda'
checkpoint_gae = torch.load(checkpoints_dir / 'gae.ckpt')
checkpoint_vgae = torch.load(checkpoints_dir / 'vgae.ckpt')
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [14]:
# Przygotowanie konfiguracji dla gae
configs_dir = project_dir / 'configs'
config_gae_path = configs_dir / 'klejda_gae_config.yaml'
config_vgae_path = configs_dir / 'klejda_vgae_config.yaml'
with open(config_gae_path) as f:
	config_gae = yaml.safe_load(f)
with open(config_vgae_path) as f:
	config_vgae = yaml.safe_load(f)

In [5]:
# # Przygotowanie środowiska Framsticks oraz DEAP
# with open("../configs/final_evolution_config.yaml", 'r') as f:
# 	evolution_config = yaml.safe_load(f)
# frams.init(
#     evolution_config['frams_path']
# )
# frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])
#
# toolbox = prepare_native_toolbox(frams_lib, evolution_config)
# # TODO: TO dodać przed uruchomieniem eksperymentu
# # toolbox.register("mutate", autoencoder_mutate, gae)
# pop = toolbox.population(n=evolution_config['pop_size'])
# hof = tools.HallOfFame(evolution_config['hof_size'])
#
# stats = tools.Statistics(lambda ind: ind.fitness.values)
# filter_feasible = lambda func, criteria: func(list(filter(is_feasible_fitness_criteria, criteria)))
# stats.register("min", lambda fit: filter_feasible(np.min, fit))
# stats.register("avg", lambda fit: filter_feasible(np.mean, fit))
# stats.register("max", lambda fit: filter_feasible(np.max, fit))
#
# postProcessingGaeResult = FramsticksPostProcessor()

In [15]:
# Wersja CMA-ES
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)
frams.init(
	evolution_config['frams_path']
)
frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])
toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)


hof = tools.HallOfFame(evolution_config['hof_size'])
stats = tools.Statistics(lambda ind: ind.fitness.values)
def safe_filter_feasible(func, criteria):
    feasible_fits = list(filter(is_feasible_fitness_criteria, criteria))
    if len(feasible_fits) == 0:
        return np.nan
    return func(feasible_fits)
stats.register("min", lambda fit: safe_filter_feasible(np.min, fit))
stats.register("avg", lambda fit: safe_filter_feasible(np.mean, fit))
stats.register("max", lambda fit: safe_filter_feasible(np.max, fit))

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Available objects: ['CheckpointEvent', 'Collision', 'CrCollision', 'Creature', 'CreatureSettings', 'CreatureSignals', 'CreatureSnapshot', 'Dictionary', 'ExpProperties', 'ExpState', 'ExtValue', 'File', 'FunctionReference', 'GenMan', 'GenManStats', 'GenePool', 'GenePools', 'Geno', 'GenoConverters', 'Genotype', 'Interface', 'Joint', 'Loader', 'Math', 'MechJoint', 'MechPart', 'MessageCatcher', 'Model', 'ModelGeometry', 'ModelSymmetry', 'Neuro', 'NeuroClass', 'NeuroClassLibrary', 'NeuroDef', 'NeuroSignals', 'NeuronsSimEnabled', 'ODE', 'Orient', 'Part', 'Pop

# Experiments

## GAE

In [16]:
gae_non_cyclic = GraphAutoencoder(config=config_gae, frams_module=frams).double()
gae_non_cyclic.load_state_dict(checkpoint_gae['state_dict'])
gae_non_cyclic.eval()
evaluator = AutoencoderEvaluator(gae_non_cyclic, frams_lib,evolution_config['opt_criteria'], evolution_config)
toolbox.register("evaluate", evaluator)

### Trained once

In [90]:
pop, log, reconstruction_ratio = run_cma_es_with_validation(
	 toolbox,
	ngen = evolution_config['generations'],
	stats=stats,
	halloffame=hof,
	verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	valid_recon_ratio	valid_frams_ratio	min        	avg     	max   
0  	500   	1                	0.106            	-0.00670985	0.231975	1.3551
1  	500   	1                	0.15             	0.000995338	0.215431	1.66966
2  	500   	1                	0.13             	-0.01      	0.182186	0.665463
3  	500   	1                	0.158            	-0.00828553	0.193295	1.05019 
4  	500   	1                	0.162            	0.012596   	0.182999	0.71374 
5  	500   	1                	0.198            	-0.00102711	0.187068	1.43553 
6  	500   	1                	0.218            	-0.00882864	0.21466 	1.32131 
7  	500   	1                	0.246            	-0.01      	0.197017	1.57608 
8  	500   	1                	0.214            	0.00769274 	0.181433	0.550206
9  	500   	1                	0.26             	-0.00172457	0.174026	1.08333 
10 	500   	1                	0.246            	-0.000419098	0.205252	2.13088 
11 	500   	1                	0.274            	-0.00622534 	0.206675	2.06886 
12

### Continual training

In [17]:
with open("../configs/klejda_gae_config.yaml") as f:
    config = yaml.safe_load(f)


for i in range(3):
	# Krok 1: Przeprowadzenie ewolucji
	toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)
	toolbox.register("evaluate", evaluator)

	pop, log, reconstruction_above_threshold, created_individuals = run_cma_es_with_validation(
		 toolbox,
		ngen = evolution_config['generations'],
		stats=stats,
		halloffame=hof,
		verbose=True,
		validity_threshold=0.5
	)

	# Krok 2: Douczenie autoenkodera
	# Zakończono ewolucję z powodu spadku jakości rekonstrukcji
	if not reconstruction_above_threshold:
		pass

	genotypes = []
	for ind in created_individuals:
		genotypes.append({'genotype': ind.genotype, 'fitness': ind.fitness.values})
	dataset = FramsticksGraphDataset(genotypes,config["max_nodes"])

	dataset_size = len(dataset)
	train_size = int(0.8 * dataset_size)
	val_size = dataset_size - train_size
	train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

	wandb_logger = WandbLogger(project="Framsticks-GAE", name="GAE-Baseline-Test", save_dir = config["save_dir"])


	train_dataloader = DataLoader(
	    train_dataset,
	    batch_size=256,
	    shuffle=True,
	    num_workers=4,
	    persistent_workers=True
	)

	val_dataloader = DataLoader(
	    val_dataset,
	    batch_size=256,
	    shuffle=False,
	    num_workers=4,
	    persistent_workers=True
	)
	trainer = pl.Trainer(
	    max_epochs=160,
	    logger=wandb_logger,
	    log_every_n_steps=5,
	    accelerator="auto",
	    devices=1
	)
	gae_non_cyclic.train()
	trainer.fit(gae_non_cyclic, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
	gae_non_cyclic.eval()
	wandb.finish()





print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	valid_recon_ratio	valid_frams_ratio	min      	avg     	max     
0  	140   	1                	0.0571429        	0.0330582	0.176977	0.517497
1  	140   	0.992857         	0.114286         	0.0192641	0.226141	0.65967 
2  	140   	1                	0.135714         	0.0214251	0.182833	0.649366
3  	140   	1                	0.178571         	0.0197566	0.15377 	0.397019
4  	140   	0.992857         	0.142857         	0.0579776	0.138767	0.348633
5  	140   	1                	0.207143         	0.0321818	0.173755	0.537943
6  	140   	1                	0.2              	0.0437661	0.16949 	0.393469
7  	140   	1                	0.178571         	0.0262759	0.181592	0.484521
8  	140   	0.992857         	0.235714         	0.0077968	0.257845	2.18846 
9  	140   	1                	0.271429         	-0.01    	0.217267	2.13832 
10 	140   	1                	0.264286         	0.00311197	0.209842	2.10298 
11 	140   	1                	0.357143         	0.0455834 	0.229845	2.09499 
12 	140   	1           

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\pytorch_lightning\loggers\wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


99 	140   	1                	1                	0.645703    	0.645712	0.645712


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │ 70.9 K │ train │     0 │
│ 1 │ fc_z             │ Linear   │    195 │ train │     0 │
│ 2 │ decoder_a        │ DecoderA │ 38.0 K │ train │     0 │
│ 3 │ decoder_x        │ DecoderX │ 34.7 K │ train │     0 │
│ 4 │ criterion        │ MSELoss  │      0 │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 143 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 143 K                                                                                                
Total estimated model params size (MB): 0.575                                                                      
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\pytorch_lightning\utilities\data.py:79: Trying to
infer the `batch_size` from an ambiguous collection. The batch size we found is 232. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\pytorch_lightning\utilities\data.py:79: Trying to
infer the `batch_size` from an ambiguous collection. The batch size we found is 58. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

`Trainer.fit` stopped: `max_epochs=160` reached.


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading config.yaml
wandb: 
wandb: Run history:
wandb:                      epoch ▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
wandb: train/locality_correlation ▁▄▅▅▅▆▆▅▆▆▆▆▆▆▆▇▆▅▇▆▇▇▇▇▇▇▇▇▇▇███▇██████
wandb:               train/loss_A ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁
wandb:               train/loss_X █▇▆▂▂▂▃▂▂▂▂▆▂▂▃▄▂▃▃▃▂▂▂▂▄▃▂▂▂▁▂▂▂▂▂▁▁▁▁▁
wandb:        train/loss_locality █▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▁▂▂▁▂▁▁▁▁▁▁▁
wandb:           train/loss_total █▇▅▄▄▃▄▃▃▄▃▃▃▂▃▂▂▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
wandb:        trainer/global_step ▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█████
wandb:   val/locality_correlation ▃▄▃▄▄▅▄▃▁▁▅▄▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█▇██▇
wandb:                 val/loss_A ▁▁▁▁▁▁▁▁▁▁▁▁▁▂█▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                 val/loss_X █▆▆▅▅▂▂▃▂▂▂▃▃▃▃▃▂▁▂▅▂▂▄▄▃▂▂▂▁▁▁▁▂▄▂▄▂▁▁▂
wandb:                         +2 ...
wandb: 
wandb: Run summary:
wandb:                      epoch 159
w


Najlepszy fitness w HoF: 2.1884587535216578
Saved '../results/test_experiment_result.json' (10)


## VGAE

### Trained once

In [102]:
vgae_non_cyclic = VariationalGraphAutoencoder(config=config_gae, frams_module=frams).double()
vgae_non_cyclic.load_state_dict(checkpoint_vgae['state_dict'])
vgae_non_cyclic.eval()
evaluator = AutoencoderEvaluator(vgae_non_cyclic, frams_lib,evolution_config['opt_criteria'], evolution_config)
toolbox.register("evaluate", evaluator)

In [103]:
pop, log, reconstruction_ratio = run_cma_es_with_validation(
	 toolbox,
	ngen = evolution_config['generations'],
	stats=stats,
	halloffame=hof,
	verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	valid_recon_ratio	valid_frams_ratio	min        	avg     	max     
0  	140   	0.992857         	0.114286         	-0.00580787	0.201027	0.427065
1  	140   	0.992857         	0.114286         	0.00484716 	0.177089	0.358182
2  	140   	1                	0.0928571        	0.0545821  	0.199954	0.35998 
3  	140   	1                	0.0642857        	0.0475225  	0.198987	0.423257
4  	140   	1                	0.135714         	0.0374334  	0.159805	0.432233
5  	140   	0.992857         	0.221429         	-0.01      	0.221437	0.878161
6  	140   	1                	0.114286         	0.00977009 	0.209522	0.597599
7  	140   	1                	0.15             	0.00956928 	0.169665	0.839867
8  	140   	1                	0.207143         	0.0171756  	0.155442	0.782663
9  	140   	1                	0.214286         	-0.00220483	0.132693	0.813791
10 	140   	1                	0.3              	0.00110815 	0.136919	0.561555
11 	140   	1                	0.378571         	-0.01      	0.171831	0.841892

KeyboardInterrupt: 

### Continual training

# Results